Adal burada öneri motorunu deneyecek.

In [ ]:
import pandas as pd # Import pandas as pd

In [ ]:
# Step 1: Read the data (Go up one directory level and access the data folder)
df_details = pd.read_csv('../data/processed/clean_ordersdetails.csv')
df_products = pd.read_csv('../data/processed/clean_products.csv')

In [ ]:
# Step 2: Filtered only the necessary columns (memory optimization)
df_details = df_details[['OrderID', 'ProductID']]
df_products = df_products[['ProductID', 'ProductName']]

In [ ]:
# Step 3: Merge tables on ProductID
df_merged = df_details.merge(df_products, on='ProductID', how='left')

In [ ]:
# Test: Let's check the first 5 rows
print(df_merged.head())

   OrderID  ProductID                    ProductName
0    10248         11                 Queso Cabrales
1    10248         42  Singaporean Hokkien Fried Mee
2    10248         72         Mozzarella di Giovanni
3    10249         14                           Tofu
4    10249         51          Manjimup Dried Apples


In [ ]:
#? Intersection matrix: We group all products by the same OrderID (groupby) and 
# spread the product names (ProductName) into columns (unstack). 
# If a product is not in that order, we fill those empty spaces with 0.

# 1. Grouping products in each order and expanding the table horizontally
basket = (df_merged.groupby(['OrderID', 'ProductName'])
          .size()
          .unstack()
          .fillna(0)) # Filling empty values (products not purchased) with 0

# 2. Function to convert counts into 1s and 0s
def encode_units(x):
    if x <= 0:
        return 0
    if x >= 1:
        return 1

# 3. Applying the encoding rule to the entire table
basket_sets = basket.map(encode_units)

# Let's check the new matrix format (test: is it correct?)
print(basket_sets.head())

ProductName  Alice Mutton  Aniseed Syrup  Boston Crab Meat  Camembert Pierrot  \
OrderID                                                                         
10248                   0              0                 0                  0   
10249                   0              0                 0                  0   
10250                   0              0                 0                  0   
10251                   0              0                 0                  0   
10252                   0              0                 0                  1   

ProductName  Carnarvon Tigers  Chai  Chang  Chartreuse verte  \
OrderID                                                        
10248                       0     0      0                 0   
10249                       0     0      0                 0   
10250                       0     0      0                 0   
10251                       0     0      0                 0   
10252                       0     0      0      

In [ ]:
# We will feed the basket_sets table into the Apriori algorithm. v1

from mlxtend.frequent_patterns import apriori, association_rules

# 1. Find product sets that appear together at least 1% of the time. 
# We limit this because processing every possible combination would crash the RAM.
# If use_colnames=True is not set, it will show meaningless numbers (0,1,2) instead of product names.
frequent_itemsets = apriori(basket_sets, min_support=0.01, use_colnames=True)

# 2. Generate the association rules (The 3 Musketeers: Support, Confidence, Lift)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 3. Sort the rules from highest to lowest based on "Lift" (the most valuable metric for us)
rules = rules.sort_values('lift', ascending=False)

# Let's check the top 10 rules for testing
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []


c:\Projects\ecommerce-sales-analytics\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [ ]:
# Feeding the basket_sets table into the Apriori algorithm. v2

from mlxtend.frequent_patterns import apriori, association_rules

# 1. Convert 1s and 0s to boolean format (Required for newer versions of mlxtend)
basket_sets_bool = basket_sets.astype(bool)

# 2. BEST PRACTICE: Adding max_len=2. 
# This focuses only on pairs of products, making the algorithm run up to 100x faster.
# We also set the min_support threshold to 0.002 (0.2%).
frequent_itemsets = apriori(basket_sets_bool, min_support=0.002, use_colnames=True, max_len=2)

# 3. Extract the rules (Filtering for Lift values greater than 1.0)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 4. Sort the rules by the highest Lift value
rules = rules.sort_values('lift', ascending=False)

# Let's check the top 10 rules!
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

                        antecedents                     consequents   support  \
858                    (Röd Kaviar)               (Mishi Kobe Niku)  0.002410   
859               (Mishi Kobe Niku)                    (Röd Kaviar)  0.002410   
326                   (Ipoh Coffee)                     (Chocolade)  0.002410   
327                     (Chocolade)                   (Ipoh Coffee)  0.002410   
309    (Northwoods Cranberry Sauce)  (Chef Anton's Cajun Seasoning)  0.003614   
308  (Chef Anton's Cajun Seasoning)    (Northwoods Cranberry Sauce)  0.003614   
533                          (Tofu)  (Grandma's Boysenberry Spread)  0.003614   
532  (Grandma's Boysenberry Spread)                          (Tofu)  0.003614   
890     (Queso Manchego La Pastora)    (Northwoods Cranberry Sauce)  0.002410   
891    (Northwoods Cranberry Sauce)     (Queso Manchego La Pastora)  0.002410   

     confidence       lift  
858    0.142857  23.714286  
859    0.400000  23.714286  
326    0.071429   9.8